# 3.7 能同時找不同關聯嗎？

# 第 3 章：Attention，自己決定看哪裡

前置：向量、矩陣乘法和 softmax。先用單 head；每個小節都畫得出矩陣。完整模型仍留到第4章。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：不能偷看明天的答案**

對角線可看；右上角未來位置禁止。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/causal.svg")))

**先想一想：**D固定、head變多，每個head維度會怎樣？

多個 head 各自讀取，再把讀迴向量接起來。可以學不同關聯，但不能光看不同顏色就宣稱某 head 有固定語意職位。

**把直覺寫成數學：**$[B,T,D]\to[B,H,T,D/H]$，每個 head 有自己的 T×T 權重。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.attention import CausalAttention

layer = CausalAttention(width=8, heads=2)
x = torch.randn(1, 4, 8)
out, cache = layer(x)
print(out.shape, "K cache", cache[0].shape)
assert out.shape == x.shape

**如何讀結果：**拆 head 是 reshape 加 transpose；要讓 D 能整除 H。手寫權重圖見 3.6。

**只改一件事：**只把 heads 改成4，核對每head的維度。
